3. Next practical exercise — Duplicate invoice detection
This is our first fraud-risk indicator. It identifies potential duplicate postings, not confirmed fraud.

What this detects: The same vendor, invoice reference and amount appearing across multiple journal documents within a company.

In [0]:
%sql
SELECT
    f.company_code,
    c.company_name,
    f.vendor_id,
    v.vendor_name,
    f.reference,
    f.amount,
    f.currency,

    COUNT(DISTINCT CONCAT(
        f.company_code, '-',
        f.fiscal_year, '-',
        f.document_number
    )) AS document_count,

    COLLECT_SET(f.document_number) AS documents

FROM ask_sarthak.gold.fact_gl_transactions f

JOIN ask_sarthak.gold.dim_vendor v
    ON f.vendor_id = v.vendor_id

JOIN ask_sarthak.gold.dim_company c
    ON f.company_code = c.company_code

WHERE f.vendor_id IS NOT NULL
  AND f.reference IS NOT NULL

GROUP BY
    f.company_code,
    c.company_name,
    f.vendor_id,
    v.vendor_name,
    f.reference,
    f.amount,
    f.currency

HAVING COUNT(DISTINCT CONCAT(
    f.company_code, '-',
    f.fiscal_year, '-',
    f.document_number
)) > 1

ORDER BY document_count DESC;


However, this doesn't prove that duplicate invoices are absent. We may have unique references in the synthetic dataset, or possible duplicates with slightly different amounts.
Step 8 — Profile invoice reference quality
Before building more detection logic, let's understand how much invoice reference data is available.

Run this query:

In [0]:
%sql
SELECT
    COUNT(*) AS total_vendor_lines,

    COUNT(reference) AS lines_with_reference,

    COUNT(DISTINCT reference) AS unique_references,

    COUNT(DISTINCT vendor_id) AS unique_vendors,

    SUM(
        CASE
            WHEN reference IS NULL OR TRIM(reference) = ''
            THEN 1
            ELSE 0
        END
    ) AS missing_references

FROM ask_sarthak.gold.fact_gl_transactions
WHERE vendor_id IS NOT NULL;


Step 9 — Check repeated references, even when amounts differ

In [0]:
%sql
SELECT
    company_code,
    vendor_id,
    reference,
    COUNT(DISTINCT document_number) AS document_count,
    COUNT(DISTINCT amount) AS different_amounts,
    COLLECT_SET(amount) AS amounts

FROM ask_sarthak.gold.fact_gl_transactions

WHERE vendor_id IS NOT NULL
  AND reference IS NOT NULL
  AND TRIM(reference) <> ''

GROUP BY
    company_code,
    vendor_id,
    reference

HAVING COUNT(DISTINCT document_number) > 1

ORDER BY document_count DESC;

Why these two checks?
- Query 1 measures the quality and coverage of invoice references.
- Query 2 identifies repeated vendor references even if the amounts differ.
This helps us distinguish exact duplicates from potential duplicates.


Step 10 — Create a duplicate invoice test dataset
We'll first create a separate test table without modifying your original Gold data.

In [0]:
%sql
CREATE OR REPLACE TABLE ask_sarthak.analytics.duplicate_invoice_test AS

SELECT *
FROM ask_sarthak.gold.fact_gl_transactions

WHERE 1 = 0;


Now insert two copies of selected vendor expense postings under new document numbers, retaining their vendor, reference and amount.

In [0]:
%sql
INSERT INTO ask_sarthak.analytics.duplicate_invoice_test

SELECT
    *,
    CASE
        WHEN document_number LIKE 'DUPTEST%'
        THEN document_number
        ELSE CONCAT('DUPTEST-', document_number)
    END AS test_document_number
FROM ask_sarthak.gold.fact_gl_transactions
WHERE 1 = 0;


Stop here: The second statement is intentionally not our final implementation. Your fact table has a fixed schema, and we should design the test table with explicit columns rather than risk an incorrect insert.

Instead, let's use a simpler, safer approach.

##Step 10A — Build a controlled duplicate test table

In [0]:
%sql
CREATE OR REPLACE TABLE ask_sarthak.analytics.duplicate_invoice_scenarios AS

SELECT
    '1000' AS company_code,
    'V0585' AS vendor_id,
    'INV-TEST-001' AS reference,
    '2025' AS fiscal_year,
    'DOC-90001' AS document_number,
    CAST(50000 AS DECIMAL(18,2)) AS invoice_amount,
    'INR' AS currency,
    'ORIGINAL' AS scenario_type

UNION ALL

SELECT
    '1000',
    'V0585',
    'INV-TEST-001',
    '2025',
    'DOC-90002',
    CAST(50000 AS DECIMAL(18,2)),
    'INR',
    'EXACT_DUPLICATE'

UNION ALL

SELECT
    '1000',
    'V0585',
    'INV-TEST-002',
    '2025',
    'DOC-90003',
    CAST(75000 AS DECIMAL(18,2)),
    'INR',
    'ORIGINAL'

UNION ALL

SELECT
    '1000',
    'V0585',
    'INV-TEST-002',
    '2025',
    'DOC-90004',
    CAST(80000 AS DECIMAL(18,2)),
    'INR',
    'AMOUNT_MISMATCH';

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM ask_sarthak.analytics.duplicate_invoice_scenarios;


Step 11 — Build the duplicate invoice detection engine

In [0]:
%sql
CREATE OR REPLACE VIEW ask_sarthak.analytics.vw_duplicate_invoice_alerts AS

WITH invoice_groups AS (
    SELECT
        company_code,
        vendor_id,
        reference,
        fiscal_year,
        currency,

        COUNT(DISTINCT document_number) AS document_count,
        COUNT(DISTINCT invoice_amount) AS amount_variations,

        MIN(invoice_amount) AS minimum_amount,
        MAX(invoice_amount) AS maximum_amount,

        COLLECT_SET(document_number) AS document_numbers

    FROM ask_sarthak.analytics.duplicate_invoice_scenarios

    GROUP BY
        company_code,
        vendor_id,
        reference,
        fiscal_year,
        currency

    HAVING COUNT(DISTINCT document_number) > 1
)

SELECT
    *,
    CASE
        WHEN amount_variations = 1
            THEN 'HIGH'
        WHEN amount_variations > 1
            THEN 'MEDIUM'
    END AS risk_level,

    CASE
        WHEN amount_variations = 1
            THEN 'Same vendor, reference and amount across multiple documents'
        WHEN amount_variations > 1
            THEN 'Same vendor and reference with different amounts'
    END AS detection_reason

FROM invoice_groups;

In [0]:
%sql
SELECT *
FROM ask_sarthak.analytics.vw_duplicate_invoice_alerts
ORDER BY risk_level, reference;


1. Evaluate the results
INV-TEST-001
HIGH

- Vendor: V0585
- Amount: ₹50,000
- Documents: DOC-90001, DOC-90002
- Reason: Same vendor, reference and amount across two documents.

INV-TEST-002
MEDIUM

- Vendor: V0585
- Amounts: ₹75,000 and ₹80,000
- Documents: DOC-90003, DOC-90004
- Reason: Same vendor and reference, but different amounts.

Your detection logic correctly distinguishes exact duplicates from amount mismatches.


2. Next — Build an alert summary
Let's create a summary that Ask Sarthak can use to report duplicate risk.


In [0]:
%sql
SELECT
    risk_level,
    COUNT(*) AS alert_count,
    SUM(document_count) AS affected_documents
FROM ask_sarthak.analytics.vw_duplicate_invoice_alerts
GROUP BY risk_level
ORDER BY
    CASE risk_level
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
        ELSE 3
    END;


2. Next — Connect duplicate alerts to Ask Sarthak
Now we'll expose the detection results to Genie so users can investigate them using natural language.

Open Genie Agents → Ask Sarthak → Configure → Sources.

Add this data asset:
ask_sarthak.analytics.vw_duplicate_invoice_alerts


Keep your existing Metric View and other sources. Don't remove anything.

After adding it, save the configuration.

Then ask Ask Sarthak:
Show all high-risk and medium-risk duplicate invoice alerts. Include vendor ID, invoice reference, document numbers, amounts and detection reason.

Share the generated SQL and answer.